In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pylab as plt

import seaborn as sns

from skspatial.objects import Line, Plane
from skspatial.plotting import plot_3d

from skspatial.objects import Line, Cylinder, Point, Points
from skspatial.plotting import plot_3d

import phasespace

import tensorflow

import os

import bisect
import numpy as np
import matplotlib.pylab as plt
import pandas as pd

import seaborn as sns

import numpy as np
from sklearn.mixture import GaussianMixture
from scipy.stats import multivariate_normal

import numpy as np
from scipy.interpolate import griddata
from scipy.integrate import quad, trapezoid
from scipy.interpolate import CubicSpline

import matplotlib.pylab as plt
from scipy import stats
from matplotlib import cm
from matplotlib.ticker import LinearLocator

from scipy.interpolate import LinearNDInterpolator

import eloss_tools


import dm_generation_tools as dgt
import detector_simulation_tools as dst
import diagnostics as dg

import earthshine_io as eio
import diagnostics_v2 as diag


import glob

import time

####################################
import warnings
# Suppress all warnings
warnings.filterwarnings("ignore")


import pickle

In [ ]:
#infile = 'OUTPUT_FILES/generated_data_depth_-8.0--4000.0_diskR_4000.0_mDM_1000.0-9000.0_mA_0.22_dmModel_floating_HIT_DETECTOR_ave_eloss__COMBINED.parquet'
#infile = 'OUTPUT_FILES/generated_data_depth_-8.0--4000.0_diskR_4000.0_mDM_10000.0-90000.0_mA_0.22_dmModel_floating_HIT_DETECTOR_ave_eloss__COMBINED.parquet'

#df = pd.read_parquet(infile)

#df

In [ ]:
#df.info()

In [ ]:
#mass = 1000
#filter = (df['M_DM'] == mass)
#
#len(df[filter])

In [ ]:
print(files[0])

dftmp = pd.read_parquet(files[0])

dftmp.columns

In [ ]:
def get_df_of_acceptances(df, energycut=[100]):
    masses = df['M_DM'].unique()
    masses
    
    org_nevents = df['total_org_nevents'].iloc[0]
    print(org_nevents, org_nevents/1e6)

    dftot = None
    for idx,ecut in enumerate(energycut):
        # Require the muons to reach CMS with some given energy
        filter = (df['efinal_mu1']>ecut)

        # Require them to hit the inner detector
        filter_hit_id = df['hit_inner_detector']==True

        #########################################################################
        vcounts = df[filter]['M_DM'].value_counts()
        dftmp = vcounts.to_frame()
        dftmp = dftmp.reset_index(names='M_DM')
        dftmp = dftmp.rename(columns={'count':f'count_ecut{ecut}'})
        
        dftmp['org_nevents'] = org_nevents
        dftmp[f'frac_ecut{ecut}'] = dftmp[f'count_ecut{ecut}']/org_nevents

        #########################################################################
        vcounts = df[filter & filter_hit_id]['M_DM'].value_counts()
        dftmp2 = vcounts.to_frame()
        dftmp2 = dftmp2.reset_index(names='M_DM')
        dftmp2 = dftmp2.rename(columns={'count':f'count_hit_id_ecut{ecut}'})
        
        dftmp2['org_nevents'] = org_nevents
        dftmp2[f'frac_hit_id_ecut{ecut}'] = dftmp2[f'count_hit_id_ecut{ecut}']/org_nevents

        
        print(ecut)

        if idx==0:
            dftot = dftmp.copy()
            dftot = pd.merge(dftot, dftmp2, on=['M_DM', 'org_nevents'], how='outer')

        else:
            #dftot = pd.concat([dftmp, dftot], join='outer', ignore_index=True)
            dftot = pd.merge(dftot, dftmp, on=['M_DM', 'org_nevents'], how='outer')
            dftot = pd.merge(dftot, dftmp2, on=['M_DM', 'org_nevents'], how='outer')
            
    
    #dftmp.columns
    #dftmp.index
    
    #dftmp.values
    #dftmp
    return dftot

In [ ]:
files = eio.summary("data", stage="combined", dm_model="core", disk_radius=40, mDM_min=10, full_paths=True)['path']

print(files)
for file in files:
    print(file)

In [ ]:
#generated_data_depth_-8.0--4000.0_diskR_4000.0_mDM_1000.0-9000.0_mA_0.22_dmModel_floating_HIT_DETECTOR_ave_eloss__COMBINED.parquet

#model = 'dmModel_momentum'
model = 'dmModel_floating'
vol = 'depth_-8.0--4000.0_diskR_4000.0'
volume = (4000-8)*(np.pi*(4000**2))

'''
vol = 'depth_-8.0--4000.0_diskR_40.0'
volume = (4000-8)*(np.pi*(40**2))
model = 'dmModel_core'
'''

print(f'volume: {volume} m^3')
print(f'volume: {volume/1e9} km^3')


eloss = 'ave_eloss'

my_dir = './OUTPUT_FILES/'
infiles = glob.glob(my_dir + f"/*{vol}*{model}*{eloss}*COMBINED.parquet")

infiles = files

#infiles = glob

dfs = []
for infile in infiles:
    
    print(infile)
    df = pd.read_parquet(infile)

    dftmp = get_df_of_acceptances(df, energycut=[10, 100, 1000])

    dfs.append(dftmp)

dfs

dfacc = pd.concat(dfs)
dfacc = dfacc.sort_values(by='M_DM')

dfacc['volume m3'] = volume 

dfacc

In [ ]:
#model = 'dmModel_floating'
#model = 'dmModel_momentum'
#model = 'dmModel_floating'
#model = 'dmModel_core'

label_tag = ''
ylim = (0,1)
if model == 'dmModel_core':
    label_tag = 'core'
    ylim = (1e-4,0.2)
elif model == 'dmModel_floating':
    label_tag = 'floating'
    ylim = (2e-9,2e-5)
elif model == 'dmModel_momentum':
    label_tag = 'mono-energetic'
    ylim = (2e-9,2e-5)

plt.figure(figsize=(8,4))
dfacc.plot(x='M_DM', y='frac_ecut10', kind='scatter', ax=plt.gca(), color='b', marker='o', s=30,  label=r'$E_{\mu}$ > 10 GeV')
dfacc.plot(x='M_DM', y='frac_ecut100', kind='scatter', ax=plt.gca(), color='r', marker='^',s=30, label='$E_{\mu}$ > 100 GeV')
dfacc.plot(x='M_DM', y='frac_ecut1000', kind='scatter', ax=plt.gca(), color='g', marker='v',s=30, label='$E_{\mu}$ > 1000 GeV')

dfacc.plot(x='M_DM', y='frac_hit_id_ecut10', kind='scatter', ax=plt.gca(), color='b', marker='s', s=30,  label=r'$E_{\mu}$ > 10 GeV (ID)')
dfacc.plot(x='M_DM', y='frac_hit_id_ecut100', kind='scatter', ax=plt.gca(), color='r', marker='P',s=30, label='$E_{\mu}$ > 100 GeV (ID)')
dfacc.plot(x='M_DM', y='frac_hit_id_ecut1000', kind='scatter', ax=plt.gca(), color='g', marker='>',s=30, label='$E_{\mu}$ > 1000 GeV (ID)')


plt.title(f'DM model: {label_tag}     volume: {volume/1e9:.3f} km$^3$')
plt.xscale('log')
plt.xlabel('$M_{DM}$ GeV/c$^2$', fontsize=16)
#plt.ylabel('# $\mu$ strike detector / # $mu$', fontsize=16)
plt.ylabel('acceptance (frac)', fontsize=16)
plt.ylim(ylim[0], ylim[1])

plt.xlim(5e2, 1e9)

plt.legend(fontsize=12)
plt.yscale('log')
plt.tight_layout()

filename = f'acc_dm_model_{label_tag}.png'
plt.savefig(filename)

filename = f'acc_dm_model_{label_tag}.parquet'
dfacc.to_parquet(filename)

# New Claude/Fable efficiences

In [ ]:
"""
acceptance_v2 -- multi-file, multi-run muon detection efficiencies for EarthShine.

Key change from the original get_df_of_acceptances: counts (numerators) and
generated totals (denominators) are SUMMED across all input files sharing the
same (rock volume, M_DM) before any fraction is computed. This makes it safe
to feed in multiple runs at the same physics point (same masses, different
run_ids) as well as files covering different masses.

Files generated over different rock volumes are kept as separate rows, keyed
by volume_m3 (plus depth_min/depth_max/disk_radius for readability), so you
can cross-check that efficiencies agree between volumes.

Volume parameters are read from the parquet footer metadata (key b"earthshine")
when present, with a fallback that parses the hive-style path, e.g.
    .../depth=m100-m8_diskR=40/combined-add8072c-....parquet
where 'm' denotes a minus sign and 'p' a decimal point.
"""

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

FOOTER_KEY = b"earthshine"

# e.g. depth=m100-m8_diskR=40  or  depth=m4000-m8_diskR=4000p5
_PATH_RE = re.compile(r"depth=(m?[0-9p]+)-(m?[0-9p]+)_diskR=(m?[0-9p]+)")


def _tok_to_float(tok):
    """'m100' -> -100.0, '4000p5' -> 4000.5"""
    sign = -1.0 if tok.startswith("m") else 1.0
    return sign * float(tok.lstrip("m").replace("p", "."))


def get_volume_params(path):
    """
    Return dict(depth_min, depth_max, disk_radius) for a file.

    Prefers the embedded footer metadata; falls back to parsing the path.
    Convention: depth_max is the numerically larger (shallower) boundary.
    """
    path = Path(path)

    # 1) Footer metadata (source of truth)
    try:
        meta = pq.read_schema(path).metadata or {}
        if FOOTER_KEY in meta:
            p = json.loads(meta[FOOTER_KEY])
            if all(p.get(k) is not None for k in ("depth_min", "depth_max", "disk_radius")):
                return {
                    "depth_min": float(p["depth_min"]),
                    "depth_max": float(p["depth_max"]),
                    "disk_radius": float(p["disk_radius"]),
                }
    except Exception:
        pass  # fall through to path parsing

    # 2) Path fallback
    m = _PATH_RE.search(str(path))
    if m is None:
        raise ValueError(
            f"Could not determine volume parameters for {path}: "
            "no 'earthshine' footer metadata and no 'depth=..._diskR=...' in path."
        )
    d1 = _tok_to_float(m.group(1))
    d2 = _tok_to_float(m.group(2))
    return {
        "depth_min": min(d1, d2),
        "depth_max": max(d1, d2),
        "disk_radius": _tok_to_float(m.group(3)),
    }


def rock_volume_m3(vp):
    """Cylinder volume in m^3 from depth_min/depth_max/disk_radius (all in m)."""
    span = vp["depth_max"] - vp["depth_min"]
    return span * np.pi * vp["disk_radius"] ** 2


def get_df_of_acceptances(files, energycut=(10, 100, 1000), verbose=True):
    """
    Compute per-mass detection efficiencies from one or many parquet files.

    Parameters
    ----------
    files : str, Path, or list of str/Path
        Input parquet file(s). May freely mix files with the same masses
        (multiple runs at one physics point) and different masses. Files
        with different rock volumes are aggregated separately.
    energycut : sequence of floats
        Energy cuts (GeV) applied to efinal_mu1.
    verbose : bool
        Print per-file summary lines.

    Returns
    -------
    DataFrame with one row per (volume, M_DM):
        depth_min, depth_max, disk_radius, volume_m3, M_DM, n_generated,
        count_ecut{E}, frac_ecut{E},
        count_hit_id_ecut{E}, frac_hit_id_ecut{E}   for each E in energycut.

    Notes
    -----
    - Fractions are computed only AFTER summing counts and generated totals
      across files, which is the statistically correct combination for
      multiple runs at the same mass point.
    - n_generated for a mass is the sum of total_org_nevents over the files
      in which that mass appears. A mass can only be discovered from the
      rows present in a file; a run in which a mass had zero surviving
      events would be invisible here (astronomically unlikely for your
      statistics, but worth knowing).
    """
    if isinstance(files, (str, Path)):
        files = [files]
    energycut = list(energycut)

    records = []
    for f in files:
        vp = get_volume_params(f)
        vol = rock_volume_m3(vp)

        df = pd.read_parquet(f)

        if verbose:
            print(f"{f}")
            print(
                f"  depth=[{vp['depth_min']}, {vp['depth_max']}] m, "
                f"diskR={vp['disk_radius']} m, volume={vol:.4g} m^3 "
                f"({vol / 1e9:.4g} km^3)"
            )

        for mass, sub in df.groupby("M_DM"):
            org_nevents = sub["total_org_nevents"].iloc[0]
            rec = {
                "depth_min": vp["depth_min"],
                "depth_max": vp["depth_max"],
                "disk_radius": vp["disk_radius"],
                "volume_m3": vol,
                "M_DM": mass,
                "n_generated": org_nevents,
            }
            hit_id = sub["hit_inner_detector"] == True  # noqa: E712 (handles NaN/object)
            for ecut in energycut:
                passed = sub["efinal_mu1"] > ecut
                rec[f"count_ecut{ecut}"] = int(passed.sum())
                rec[f"count_hit_id_ecut{ecut}"] = int((passed & hit_id).sum())
            records.append(rec)

            if verbose:
                print(f"    M_DM={mass}: n_generated={org_nevents}")

    per_file = pd.DataFrame(records)

    group_keys = ["depth_min", "depth_max", "disk_radius", "volume_m3", "M_DM"]
    sum_cols = ["n_generated"] + [c for c in per_file.columns if c.startswith("count_")]
    dfacc = per_file.groupby(group_keys, as_index=False)[sum_cols].sum()

    # Fractions computed only after summation
    for ecut in energycut:
        dfacc[f"frac_ecut{ecut}"] = dfacc[f"count_ecut{ecut}"] / dfacc["n_generated"]
        dfacc[f"frac_hit_id_ecut{ecut}"] = (
            dfacc[f"count_hit_id_ecut{ecut}"] / dfacc["n_generated"]
        )

    return dfacc.sort_values(["volume_m3", "M_DM"]).reset_index(drop=True)


def compare_across_volumes(dfacc, frac_col):
    """
    Cross-check helper: pivot one fraction column so each rock volume is a
    column, indexed by M_DM. Consistent physics should give consistent
    efficiencies across volumes (within Poisson errors).
    """
    return dfacc.pivot_table(index="M_DM", columns="volume_m3", values=frac_col)

In [ ]:
#files = eio.summary("data", stage="combined", dm_model="core", disk_radius=40, mDM_min=10, full_paths=True)['path']
files = eio.summary("data", stage="combined", dm_model="core", mDM_min=10, full_paths=True)['path']


print(files)
for file in files:
    print(file)

In [ ]:
dfacc = get_df_of_acceptances(files, energycut=[10, 100, 1000])

dfacc

In [ ]:
compare_across_volumes(dfacc, "frac_hit_id_ecut10")


# Checking efficiences for different parameter settings

In [ ]:
cat = eio.read_catalog("data")

#depth = -108.0
depth = 100

df, params = eio.load_many(cat, dm_model="momentum_constrained", stage="combined",
                      depth_min=depth, depth_max=depth, \
                     mDM_min=10000, mDM_max=10000, eloss='ave')   # add filters until exactly 1 matches

norg = df['total_org_nevents'].iloc[0]
filter = df['hit_inner_detector']==True
n = len(df[filter])

print(f'depth: {depth}   # org: {norg}    n: {n}      n/# org: {n/norg:.2e}      100*n/# org (%): {100*n/norg:.2e}  ')

In [ ]:
filter = (df['efinal_mu1']>10)


sns.histplot(df[filter], x='distance_to_detector', bins=100, hue='M_DM')


In [ ]:
filter = (df['efinal_mu1']>10)

sns.histplot(df[filter], x='y0', bins=100, hue='M_DM')


In [ ]:
filter = (df['efinal_mu1']>100)

sns.histplot(df[filter], x='y0', bins=100, hue='M_DM', binrange=(-4000,0))


In [ ]:
filter = (df['M_DM']==10000) & (df['efinal_mu1']>10)

df[filter]['y0'].hist(bins=100)